# Optional Colab source staging

This notebook is not needed for local use and has not been validated as a cloud inference run. It never authorizes uploading participant data. The bundled codebooks use fictional examples. Read docs/reproducibility.md and docs/data-governance.md first. The current pipeline has per-label SQLite checkpointing; logs and outputs remain private.


In [ ]:
from pathlib import Path
import hashlib, json, zipfile
RUN_STAGING = False  # change only after approving this cloud storage/runtime
BUNDLE = Path('/content/ptg-colab.zip')
PROJECT = Path('/content/ptg-source')
CONFIG = 'config/latest-local.yaml'
AUDIT_SCRIPT = 'scripts/audit_workbook.py'
RESERVED_PERIOD = 'FA26'  # never use as all-negative truth or for prompt selection


In [ ]:
if RUN_STAGING:
    assert BUNDLE.is_file(), 'Supply the sanitized source-only bundle separately'
    assert not PROJECT.exists(), 'Choose a new staging directory'
    with zipfile.ZipFile(BUNDLE) as archive:
        manifest = json.loads(archive.read('manifest.json'))
        expected = set(manifest['files']) | {'manifest.json'}
        assert set(archive.namelist()) == expected, 'Unexpected archive entries'
        for name, digest in manifest['files'].items():
            relative = Path(name)
            assert not relative.is_absolute() and '..' not in relative.parts
            assert not name.startswith(('data/', 'artifacts/')) and '.env' not in name
            payload = archive.read(name)
            assert hashlib.sha256(payload).hexdigest() == digest
            dest = PROJECT / relative
            dest.parent.mkdir(parents=True, exist_ok=True)
            dest.write_bytes(payload)
    print('Source-only staging complete; no transcript was uploaded by this notebook.')
else:
    print('Cloud staging is disabled. Use the local synthetic demo instead.')


## After approved staging

Install dependencies with `uv sync --frozen` in the staged checkout. A model server and privately supplied approved workbook/codebook are separate prerequisites. No embeddings, chat inference or FA26 scoring are launched here. Use the local reproduction guide for the actual pipeline commands; do not save participant text in notebook cells or outputs.
